# Gemma 4 31B Tool-Calling Fine-Tune V2

Based on the working AMD Dev Cloud notebook and the official Unsloth AMD / Gemma 4 docs.

Primary dataset: [mtita/gemma4-tool-calling-training](https://huggingface.co/datasets/mtita/gemma4-tool-calling-training)

**Validated candidate additions for V2**
- `togethercomputer/CoderForge-Preview` — coding-agent trajectories
- `nebius/SWE-agent-trajectories` — SWE tool-use trajectories
- `Salesforce/xlam-function-calling-60k` — function calling (**gated terms**)
- `NousResearch/hermes-function-calling-v1` — function calling / JSON mode
- `microsoft/orca-agentinstruct-1M-v1` — large agent-style instruction corpus

**Recommended V2 mix**
- 35-40% coding-agent trajectories
- 20-25% function-calling data
- 20-25% current code/tool-use corpus
- 10-15% concise preference / anti-verbosity data
- 5-10% no-tool / brief-answer negatives


### Installation


### V2 dataset notes

This notebook keeps the **same working training path** as the base notebook, but records the validated V2 data directions.

**Availability notes**
- `togethercomputer/CoderForge-Preview`: available on Hugging Face
- `nebius/SWE-agent-trajectories`: available on Hugging Face
- `Salesforce/xlam-function-calling-60k`: available, but requires accepting dataset terms
- `NousResearch/hermes-function-calling-v1`: available on Hugging Face
- `microsoft/orca-agentinstruct-1M-v1`: available on Hugging Face

**RL note from the Unsloth RL guide**
- RL / GRPO is best used **after** stable SFT
- For this project, RL should target **conciseness / non-repetition / correct tool choice**, not replace SFT
- Use a rubric of small verifiable rewards instead of one giant reward


In [ ]:
# Validated V2 candidate datasets
VALIDATED_DATASETS = {
    "current_base": {"dataset": "mtita/gemma4-tool-calling-training", "note": "current working base"},
    "coderforge": {"dataset": "togethercomputer/CoderForge-Preview", "note": "coding-agent trajectories"},
    "swe_agent": {"dataset": "nebius/SWE-agent-trajectories", "note": "SWE bash/edit/search trajectories"},
    "xlam_fc": {"dataset": "Salesforce/xlam-function-calling-60k", "note": "gated function-calling dataset"},
    "hermes_fc": {"dataset": "NousResearch/hermes-function-calling-v1", "note": "function-calling / JSON mode"},
    "orca_agent": {"dataset": "microsoft/orca-agentinstruct-1M-v1", "note": "large agentic instruction corpus"},
}
for name, info in VALIDATED_DATASETS.items():
    print(f"{name}: {info['dataset']}  -- {info['note']}")


In [ ]:
# === Create a clean AMD/Unsloth kernel (doc-aligned path) ===
# This follows the official AMD + Unsloth guidance: isolated env, ROCm PyTorch, then Unsloth.
import os, re, shutil, subprocess, sys
from pathlib import Path
VENV = os.path.expanduser('~/unsloth_gemma4_env')
PY = os.path.join(VENV, 'bin', 'python')
PIP = [PY, '-m', 'pip']

def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip():
        out = r.stdout.strip().splitlines()
        for line in out[-8:]:
            print(line)
    if r.returncode != 0:
        err = r.stderr.strip() or '(no stderr)'
        raise RuntimeError(err)

def detect_rocm_tag():
    candidates = []
    try:
        r = subprocess.run(['amd-smi', 'version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    try:
        candidates.append(Path('/opt/rocm/.info/version').read_text())
    except Exception:
        pass
    try:
        r = subprocess.run(['hipconfig', '--version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    for text in candidates:
        m = re.search(r'(?:ROCm version: |HIP version: )([0-9]+)\.([0-9]+)', text)
        if not m:
            m = re.search(r'([0-9]+)\.([0-9]+)', text)
        if m:
            return f'rocm{m.group(1)}.{m.group(2)}'
    return 'rocm7.0'

rocm_tag = detect_rocm_tag()
print('ROCm wheel tag:', rocm_tag)

if os.path.isdir(VENV):
    shutil.rmtree(VENV)
run([sys.executable, '-m', 'venv', VENV])
run(PIP + ['install', '--upgrade', 'pip', 'setuptools', 'wheel', 'ipykernel'])
run(PIP + ['install', '--upgrade', '--force-reinstall', 'torch', 'torchvision', 'torchaudio', '--index-url', f'https://download.pytorch.org/whl/{rocm_tag}'])
run(PIP + ['install', '--no-deps', 'unsloth', 'unsloth-zoo'])
run(PIP + ['install', '--no-deps', 'git+https://github.com/unslothai/unsloth-zoo.git'])
run(PIP + ['install', 'unsloth[amd] @ git+https://github.com/unslothai/unsloth'])
run(PIP + ['install', '--upgrade', 'git+https://github.com/huggingface/transformers.git'])
run(PIP + ['install', '--upgrade', 'timm', 'torchcodec'])
run([PY, '-m', 'ipykernel', 'install', '--user', '--name', 'unsloth-gemma4-amd', '--display-name', 'Python (unsloth-gemma4-amd)'])

print('\nKernel created: Python (unsloth-gemma4-amd)')
print('Next: Kernel -> Change Kernel -> Python (unsloth-gemma4-amd)')
print('Then restart kernel and run Cell 3.')


In [ ]:
# === Verify you are on the clean Unsloth AMD kernel ===
import os, sys
VENV = os.path.expanduser('~/unsloth_gemma4_env')
print('sys.executable:', sys.executable)
print('sys.prefix:', sys.prefix)
print('VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))
on_expected_kernel = (
    sys.prefix == VENV
    or sys.executable.startswith(VENV + os.sep)
    or os.environ.get('VIRTUAL_ENV') == VENV
)
if not on_expected_kernel:
    raise RuntimeError(
        'Wrong kernel. Change kernel to Python (unsloth-gemma4-amd), then restart and re-run this cell.'
    )

import unsloth, unsloth_zoo, huggingface_hub, regex, transformers
from transformers import AutoConfig

print(f'unsloth: {getattr(unsloth, "__version__", "OK")} ({unsloth.__file__})')
print(f'unsloth_zoo: {getattr(unsloth_zoo, "__version__", "OK")} ({unsloth_zoo.__file__})')
print(f'huggingface_hub: {huggingface_hub.__version__} ({huggingface_hub.__file__})')
print(f'regex: {regex.__version__} ({regex.__file__})')
print(f'transformers: {transformers.__version__} ({transformers.__file__})')
c = AutoConfig.from_pretrained('unsloth/gemma-4-31B-it')
print(f'AutoConfig OK: model_type={c.model_type}')

import torch; torch._dynamo.config.recompile_limit = 64
print('Next: run the HF token cell, then the Load Model cell.')


In [ ]:
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"  # <-- paste your token here


### Load Model


In [ ]:
from unsloth import FastModel
import torch, os

HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # LoRA fine-tuning (much less VRAM)
    token = HF_TOKEN,
)


### Add LoRA Adapters


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Text-only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 32,           # Higher rank for tool-calling precision
    lora_alpha = 32,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)


### Data Prep

This V2 notebook still trains on the **current stable dataset** by default, while documenting validated candidate additions for the next curation pass.

We use the `gemma-4` chat template. Gemma 4 renders conversations like:
```
<bos><|turn>user
Hello<turn|>
<|turn>model
Hey there!<turn|>
```


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)


In [ ]:
from datasets import load_dataset

# Keep the working base dataset here until the V2 curated mix is assembled.
ACTIVE_DATASET = "mtita/gemma4-tool-calling-training"
ACTIVE_FILE = "combined_training.jsonl"

dataset = load_dataset(
    ACTIVE_DATASET,
    data_files=ACTIVE_FILE,
    split="train",
)
print(f"Loaded {len(dataset)} training examples from {ACTIVE_DATASET}/{ACTIVE_FILE}")
print(dataset[0]["messages"][:2])


Map roles for Gemma 4 (system → user prefix, tool → user prefix) and apply chat template.


In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        formatted = []
        for msg in messages:
            role = msg["role"]
            content = msg.get("content", "")
            if role == "system":
                formatted.append({"role": "user", "content": f"[System Instructions]\n{content}"})
                formatted.append({"role": "assistant", "content": "Understood."})
            elif role == "user":
                formatted.append({"role": "user", "content": content})
            elif role == "assistant":
                formatted.append({"role": "assistant", "content": content})
            elif role == "tool":
                tool_name = msg.get("name", "tool")
                formatted.append({"role": "user", "content": f"[Tool Result: {tool_name}]\n{content}"})
        try:
            text = tokenizer.apply_chat_template(
                formatted, tokenize=False, add_generation_prompt=False
            ).removeprefix('<bos>')
            texts.append(text)
        except Exception:
            texts.append("")
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.filter(lambda x: len(x["text"]) > 50)
print(f"Formatted: {len(dataset)} examples")
print(dataset[0]["text"][:500])


### RL / GRPO next step (not part of this notebook)

Per the Unsloth RL guide, RL should come **after stable SFT**. For this project, RL is best used later to reward:
- concise answers
- correct tool selection
- low repetition
- stopping when the task is done

Do not switch this notebook to RL yet; use it to produce the stronger SFT base first.


### Train the model


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)


Only train on assistant outputs, ignore user inputs:


In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)


In [ ]:
# Verify masking - should see full conversation
tokenizer.decode(trainer.train_dataset[0]["input_ids"])


In [ ]:
# Verify masking - should see ONLY assistant responses (rest is spaces)
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


### Train!


In [ ]:
trainer_stats = trainer.train()


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")


### Test Inference


In [ ]:
messages = [{
    "role": "user",
    "content": [{"type": "text", "text": """You have these tools:\n"
"- read(path): Read a file\n"
"- edit(path, old_text, new_text): Replace text in a file\n"
"- bash(command): Run a shell command\n\n"
"Fix the typo in src/utils.py where 'retrun' should be 'return'."""}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 512,
    use_cache = False,
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)


### Save LoRA adapters

**Set your HF token below!**


In [ ]:
model.save_pretrained("gemma_4_lora")
tokenizer.save_pretrained("gemma_4_lora")

model.push_to_hub("mtita/gemma4-31b-tool-calling-lora", token=HF_TOKEN)
tokenizer.push_to_hub("mtita/gemma4-31b-tool-calling-lora", token=HF_TOKEN)


### Export to GGUF

Supported: q4_k_m (recommended), q5_k_m, q8_0, f16, bf16 and more.


In [ ]:
model.save_pretrained_gguf(
    "gemma_4_finetune",
    tokenizer,
    quantization_method = "q4_k_m",
)


In [ ]:
# Push GGUF to HuggingFace
model.push_to_hub_gguf(
    "mtita/gemma4-31b-tool-calling-q4_k_m-GGUF",
    tokenizer,
    quantization_method = "q4_k_m",
    token = HF_TOKEN,
)
